# Olist e-commerce analytics

SQL and pandas analysis of ~99k orders from Olist, a Brazilian marketplace (Kaggle, CC BY-NC-SA 4.0).
The question: where should a marketplace manager focus: delivery, seller quality or customer retention?

**Contents**

1. Raw data profile and data quality checks
2. Cleaning, loading into SQLite, and validation (the notebook stops if any check fails)
3. SQL analysis, headline metrics and metric definitions (the SQL is in `queries.sql`, shared with the dashboard)
4. Cohort retention
5. Sizing the recommendations
6. Memo: three actions

Run top to bottom from the project folder. The raw CSVs go in `data/`; the dashboard (`streamlit run app.py`)
reads the SQLite database this notebook builds.

In [1]:
import re
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from dataclasses import dataclass
import pandas as pd
import sqlite3
from collections.abc import Callable
import numpy as np
import re
import json

# Everything lives in data/: the raw CSVs, the SQLite database and the saved query results.
DATA = Path("data")
RAW_DIR = PROCESSED_DIR = DATA
DB_PATH = DATA / "olist.db"
QUERY_FILE = Path("queries.sql")
METRIC_NAMES = ["revenue", "active_customers", "repeat_customers", "late_delivery_rate", "avg_review_score"]

# Logical table name -> raw CSV file name.
RAW_FILES: dict[str, str] = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# Columns that hold timestamps, per table.
DATE_COLUMNS: dict[str, list[str]] = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

# Zip prefixes have leading zeros, so they must be read as text.
STRING_COLUMNS = {
    "customer_zip_code_prefix": str,
    "seller_zip_code_prefix": str,
    "geolocation_zip_code_prefix": str,
}


def load_queries(path=QUERY_FILE) -> dict[str, str]:
    """Split queries.sql into {name: sql} using '-- name: x' markers."""
    parts = re.split(r"^-- name: (\w+)\s*$", path.read_text(encoding="utf-8"), flags=re.MULTILINE)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}


def run_query(name: str, con: sqlite3.Connection | None = None) -> pd.DataFrame:
    """Run one named query from queries.sql and return a DataFrame."""
    sql = load_queries()[name]
    if con is None:
        with sqlite3.connect(DB_PATH) as c:
            return pd.read_sql_query(sql, c)
    return pd.read_sql_query(sql, con)


run_metric = run_query  # metric queries live in the same file


def show(markdown: str) -> None:
    """Render a generated report as markdown in the notebook."""
    display(Markdown(markdown))

## Part 1 · Raw data profile and quality checks


Before cleaning anything, this notebook answers two questions:

1. *What is in each table?* Row counts, duplicate keys, nulls and date ranges.
2. *What is wrong with it?* Impossible dates, orphan rows, duplicates, odd values and missing values,
   each paired with the rule the cleaning notebook (Part 2) applies.

Every number in both reports is computed here; nothing is typed by hand.

### Part 1: profile every raw table

The report records, per table: row count, duplicate rows, duplicate keys,
null counts per column and the date range of timestamp columns. It is
generated from the raw CSVs so the cleaning rules can be justified by it.

In [2]:
PRIMARY_KEYS: dict[str, list[str]] = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}

In [3]:
def read_raw(table: str) -> pd.DataFrame:
    """Read one raw CSV with zip codes as text and the BOM stripped."""
    return pd.read_csv(RAW_DIR / RAW_FILES[table], dtype=STRING_COLUMNS, encoding="utf-8-sig")

In [4]:
def profile_table(table: str, df: pd.DataFrame) -> list[str]:
    """Return markdown lines describing one table."""
    lines = [f"## {table}", "", f"- Rows: {len(df):,}", f"- Columns: {df.shape[1]}"]
    lines.append(f"- Fully duplicated rows: {int(df.duplicated().sum()):,}")
    key = PRIMARY_KEYS.get(table)
    if key:
        lines.append(f"- Duplicate keys on {key}: {int(df.duplicated(subset=key).sum()):,}")
    for col in DATE_COLUMNS.get(table, []):
        parsed = pd.to_datetime(df[col], errors="coerce")
        lines.append(f"- `{col}` range: {parsed.min()} to {parsed.max()}")
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        lines.append("- Nulls: none")
    else:
        lines.append("- Nulls:")
        for col, n in nulls.items():
            lines.append(f"  - `{col}`: {int(n):,} ({n / len(df):.1%})")
    if table == "orders":
        lines.append("- `order_status` counts:")
        for status, n in df["order_status"].value_counts().items():
            lines.append(f"  - {status}: {n:,}")
    if table == "order_reviews":
        lines.append(
            f"- Orders with more than one review: {int(df['order_id'].duplicated().sum()):,} extra review rows"
        )
    lines.append("")
    return lines

#### Run the profile

In [5]:
out = ["# Raw data profile", "", "Generated from the raw CSVs.", ""]
tables = {t: read_raw(t) for t in RAW_FILES}
for table, df in tables.items():
    out.extend(profile_table(table, df))

# Cross-table checks that matter for the analysis.
products, translation = tables["products"], tables["category_translation"]
cats = set(products["product_category_name"].dropna())
missing = sorted(cats - set(translation["product_category_name"]))
cust = tables["customers"]
out += [
    "## Cross-table checks",
    "",
    f"- Distinct `customer_id`: {cust['customer_id'].nunique():,}; "
    f"distinct `customer_unique_id`: {cust['customer_unique_id'].nunique():,}",
    f"- Product categories with no English translation: {missing}",
    f"- Orders with no items: {(~tables['orders']['order_id'].isin(tables['order_items']['order_id'])).sum():,}",
    f"- Orders with no payment: {(~tables['orders']['order_id'].isin(tables['order_payments']['order_id'])).sum():,}",
    "",
]
show("\n".join(out))

# Raw data profile

Generated from the raw CSVs.

## customers

- Rows: 99,441
- Columns: 5
- Fully duplicated rows: 0
- Duplicate keys on ['customer_id']: 0
- Nulls: none

## geolocation

- Rows: 1,000,163
- Columns: 5
- Fully duplicated rows: 261,831
- Nulls: none

## order_items

- Rows: 112,650
- Columns: 7
- Fully duplicated rows: 0
- Duplicate keys on ['order_id', 'order_item_id']: 0
- `shipping_limit_date` range: 2016-09-19 00:15:34 to 2020-04-09 22:35:08
- Nulls: none

## order_payments

- Rows: 103,886
- Columns: 5
- Fully duplicated rows: 0
- Duplicate keys on ['order_id', 'payment_sequential']: 0
- Nulls: none

## order_reviews

- Rows: 99,224
- Columns: 7
- Fully duplicated rows: 0
- Duplicate keys on ['review_id']: 814
- `review_creation_date` range: 2016-10-02 00:00:00 to 2018-08-31 00:00:00
- `review_answer_timestamp` range: 2016-10-07 18:32:28 to 2018-10-29 12:27:35
- Nulls:
  - `review_comment_title`: 87,656 (88.3%)
  - `review_comment_message`: 58,247 (58.7%)
- Orders with more than one review: 551 extra review rows

## orders

- Rows: 99,441
- Columns: 8
- Fully duplicated rows: 0
- Duplicate keys on ['order_id']: 0
- `order_purchase_timestamp` range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18
- `order_approved_at` range: 2016-09-15 12:16:38 to 2018-09-03 17:40:06
- `order_delivered_carrier_date` range: 2016-10-08 10:34:01 to 2018-09-11 19:48:28
- `order_delivered_customer_date` range: 2016-10-11 13:46:32 to 2018-10-17 13:22:46
- `order_estimated_delivery_date` range: 2016-09-30 00:00:00 to 2018-11-12 00:00:00
- Nulls:
  - `order_approved_at`: 160 (0.2%)
  - `order_delivered_carrier_date`: 1,783 (1.8%)
  - `order_delivered_customer_date`: 2,965 (3.0%)
- `order_status` counts:
  - delivered: 96,478
  - shipped: 1,107
  - canceled: 625
  - unavailable: 609
  - invoiced: 314
  - processing: 301
  - created: 5
  - approved: 2

## products

- Rows: 32,951
- Columns: 9
- Fully duplicated rows: 0
- Duplicate keys on ['product_id']: 0
- Nulls:
  - `product_category_name`: 610 (1.9%)
  - `product_name_lenght`: 610 (1.9%)
  - `product_description_lenght`: 610 (1.9%)
  - `product_photos_qty`: 610 (1.9%)
  - `product_weight_g`: 2 (0.0%)
  - `product_length_cm`: 2 (0.0%)
  - `product_height_cm`: 2 (0.0%)
  - `product_width_cm`: 2 (0.0%)

## sellers

- Rows: 3,095
- Columns: 4
- Fully duplicated rows: 0
- Duplicate keys on ['seller_id']: 0
- Nulls: none

## category_translation

- Rows: 71
- Columns: 2
- Fully duplicated rows: 0
- Duplicate keys on ['product_category_name']: 0
- Nulls: none

## Cross-table checks

- Distinct `customer_id`: 99,441; distinct `customer_unique_id`: 96,096
- Product categories with no English translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
- Orders with no items: 775
- Orders with no payment: 1


---
### Part 2: data quality issues

Each check below counts the rows that break one expectation. Checks that find 0 rows are kept in the
report on purpose: they are the rules Part 2 enforces on every run.

#### The checks

Each issue is paired with the rule the pipeline applies to it (see Part 2
and the query conventions in `queries.sql`).

In [6]:
WINDOW_START = pd.Timestamp("2017-01-01")
WINDOW_END = pd.Timestamp("2018-09-01")

In [7]:
@dataclass
class Issue:
    """One data quality finding."""

    category: str
    table: str
    check: str
    rows: int
    handling: str

In [8]:
def load_raw() -> dict[str, pd.DataFrame]:
    """Read every raw CSV with dates parsed and zip prefixes kept as text."""
    out = {}
    for table, file in RAW_FILES.items():
        df = pd.read_csv(RAW_DIR / file, dtype=STRING_COLUMNS, encoding="utf-8-sig")
        for col in DATE_COLUMNS.get(table, []):
            df[col] = pd.to_datetime(df[col], errors="coerce")
        out[table] = df
    return out

In [9]:
def date_issues(raw: dict[str, pd.DataFrame]) -> list[Issue]:
    """Timestamps that are impossible or out of order."""
    o = raw["orders"]
    purchase = o["order_purchase_timestamp"]
    approved = o["order_approved_at"]
    carrier = o["order_delivered_carrier_date"]
    delivered = o["order_delivered_customer_date"]
    estimated = o["order_estimated_delivery_date"]
    reviews = raw["order_reviews"].merge(o[["order_id", "order_purchase_timestamp"]], on="order_id")
    not_used = "Kept. Carrier and approval dates are not used in any metric."
    return [
        Issue(
            "Impossible dates",
            "orders",
            "Delivered to customer before purchase",
            int((delivered < purchase).sum()),
            "Checked in validation (Part 2): the pipeline stops if this is ever > 0.",
        ),
        Issue(
            "Impossible dates",
            "orders",
            "Estimated delivery date before purchase",
            int((estimated < purchase).sum()),
            "Checked in validation (Part 2): the pipeline stops if this is ever > 0.",
        ),
        Issue(
            "Impossible dates",
            "orders",
            "Approved before purchase",
            int((approved < purchase).sum()),
            "Checked in validation (Part 2): the pipeline stops if this is ever > 0.",
        ),
        Issue(
            "Inverted dates", "orders", "Handed to carrier before purchase", int((carrier < purchase).sum()), not_used
        ),
        Issue(
            "Inverted dates",
            "orders",
            "Handed to carrier before payment approval",
            int((carrier < approved).sum()),
            not_used,
        ),
        Issue(
            "Inverted dates",
            "orders",
            "Delivered to customer before handed to carrier",
            int((delivered < carrier).sum()),
            "Kept. Delivery time is measured from purchase, which is never after delivery.",
        ),
        Issue(
            "Inverted dates",
            "order_reviews",
            "Review created before the order's purchase date",
            int((reviews["review_creation_date"] < reviews["order_purchase_timestamp"].dt.normalize()).sum()),
            "Kept. The score is still used; review timing is not analysed.",
        ),
        Issue(
            "Inconsistent status",
            "orders",
            "Status 'delivered' but no delivery date",
            int(((o["order_status"] == "delivered") & delivered.isna()).sum()),
            "Delay is NULL, so these orders drop out of late-delivery metrics.",
        ),
        Issue(
            "Inconsistent status",
            "orders",
            "Not delivered (e.g. canceled) but has a delivery date",
            int(((o["order_status"] != "delivered") & delivered.notna()).sum()),
            "Delivery metrics set to NULL during cleaning (Part 2).",
        ),
        Issue(
            "Out of window",
            "orders",
            "Purchased before 2017-01 or after 2018-08",
            int(((purchase < WINDOW_START) | (purchase >= WINDOW_END)).sum()),
            "Kept in the database, excluded from every metric (in_window = 0).",
        ),
    ]

In [10]:
def orphan_issues(raw: dict[str, pd.DataFrame]) -> list[Issue]:
    """Rows whose key has no match in the parent or child table."""
    o, items, pay = raw["orders"], raw["order_items"], raw["order_payments"]
    rev, prod, sel, cust = raw["order_reviews"], raw["products"], raw["sellers"], raw["customers"]
    fk_rule = "Checked in validation (Part 2): the pipeline stops if this is ever > 0."
    no_items = o[~o["order_id"].isin(items["order_id"])]
    n_dead = int(no_items["order_status"].isin(["canceled", "unavailable"]).sum())
    return [
        Issue(
            "Orphan rows",
            "order_items",
            "order_id not in orders",
            int((~items["order_id"].isin(o["order_id"])).sum()),
            fk_rule,
        ),
        Issue(
            "Orphan rows",
            "order_items",
            "product_id not in products",
            int((~items["product_id"].isin(prod["product_id"])).sum()),
            fk_rule,
        ),
        Issue(
            "Orphan rows",
            "order_items",
            "seller_id not in sellers",
            int((~items["seller_id"].isin(sel["seller_id"])).sum()),
            fk_rule,
        ),
        Issue(
            "Orphan rows",
            "order_payments",
            "order_id not in orders",
            int((~pay["order_id"].isin(o["order_id"])).sum()),
            fk_rule,
        ),
        Issue(
            "Orphan rows",
            "order_reviews",
            "order_id not in orders",
            int((~rev["order_id"].isin(o["order_id"])).sum()),
            fk_rule,
        ),
        Issue(
            "Orphan rows",
            "orders",
            "customer_id not in customers",
            int((~o["customer_id"].isin(cust["customer_id"])).sum()),
            fk_rule,
        ),
        Issue(
            "Missing children",
            "orders",
            "Order has no items",
            len(no_items),
            "Kept. Revenue joins to items, so these orders add no revenue "
            f"({n_dead} of them are canceled/unavailable).",
        ),
        Issue(
            "Missing children",
            "orders",
            "Order has no payment",
            int((~o["order_id"].isin(pay["order_id"])).sum()),
            "Kept. Revenue is item price, not payments.",
        ),
        Issue(
            "Missing children",
            "orders",
            "Order has no review",
            int((~o["order_id"].isin(rev["order_id"])).sum()),
            "Kept. Review averages use COUNT(review_score), so unreviewed orders are not counted as 0.",
        ),
    ]

In [11]:
def duplicate_issues(raw: dict[str, pd.DataFrame]) -> list[Issue]:
    """Duplicate reviews and other repeated keys."""
    rev, geo = raw["order_reviews"], raw["geolocation"]
    dup_ids = rev[rev["review_id"].duplicated(keep=False)]
    return [
        Issue(
            "Duplicate reviews",
            "order_reviews",
            "Extra reviews for an order that already has one",
            int(rev["order_id"].duplicated().sum()),
            "Keep the most recently answered review per order.",
        ),
        Issue(
            "Duplicate reviews",
            "order_reviews",
            "Rows whose review_id is shared with another order",
            len(dup_ids),
            f"Kept. {dup_ids['review_id'].nunique():,} review_ids each cover several orders "
            "(one review for a multi-order purchase), so reviews are keyed by order_id.",
        ),
        Issue(
            "Duplicate rows",
            "geolocation",
            "Fully duplicated rows",
            int(geo.duplicated().sum()),
            "Collapsed to one median point per zip prefix.",
        ),
    ]

In [12]:
def value_issues(raw: dict[str, pd.DataFrame]) -> list[Issue]:
    """Values that are zero, negative or unlabelled."""
    items, pay, prod = raw["order_items"], raw["order_payments"], raw["products"]
    translated = set(raw["category_translation"]["product_category_name"])
    cats = prod["product_category_name"].dropna().unique()
    return [
        Issue(
            "Suspicious values",
            "order_items",
            "Price <= 0",
            int((items["price"] <= 0).sum()),
            "Checked in validation (Part 2): the pipeline stops if this is ever > 0.",
        ),
        Issue(
            "Suspicious values",
            "order_items",
            "Freight = 0",
            int((items["freight_value"] == 0).sum()),
            "Kept (free shipping is plausible). Revenue excludes freight anyway.",
        ),
        Issue(
            "Suspicious values",
            "order_payments",
            "payment_value <= 0",
            int((pay["payment_value"] <= 0).sum()),
            "Kept and logged, so payment totals still reconcile.",
        ),
        Issue(
            "Suspicious values",
            "order_payments",
            "payment_type 'not_defined'",
            int((pay["payment_type"] == "not_defined").sum()),
            "Kept and logged.",
        ),
        Issue(
            "Suspicious values",
            "order_payments",
            "0 instalments",
            int((pay["payment_installments"] == 0).sum()),
            "Kept. Instalments only appear in the payment-mix average.",
        ),
        Issue(
            "Suspicious values",
            "products",
            "Weight = 0 g",
            int((prod["product_weight_g"] == 0).sum()),
            "Kept. Product weight is not used.",
        ),
        Issue(
            "Unlabelled values",
            "products",
            "Category has no English translation",
            int(len([c for c in cats if c not in translated])),
            "Two missing translations added by hand during cleaning (Part 2).",
        ),
    ]

In [13]:
def missing_issues(raw: dict[str, pd.DataFrame]) -> list[Issue]:
    """Every column with missing values, with the rule that applies."""
    rules = {
        ("orders", "order_approved_at"): "Kept. Not used in metrics.",
        ("orders", "order_delivered_carrier_date"): "Kept. Not used in metrics.",
        ("orders", "order_delivered_customer_date"): "Order drops out of delivery metrics (delay is NULL).",
        ("order_reviews", "review_comment_title"): "Kept. Text is optional; only the score is used.",
        ("order_reviews", "review_comment_message"): "Kept. Text is optional; only the score is used.",
        ("products", "product_category_name"): "Labelled 'unknown'.",
    }
    out = []
    for table, df in raw.items():
        for col, n in df.isna().sum().items():
            if n:
                rule = rules.get((table, col), "Kept. Column not used in metrics.")
                out.append(Issue("Missing values", table, f"`{col}` is empty ({100 * n / len(df):.1f}%)", int(n), rule))
    return out

In [14]:
def find_issues(raw: dict[str, pd.DataFrame]) -> list[Issue]:
    """All checks, in report order."""
    return date_issues(raw) + orphan_issues(raw) + duplicate_issues(raw) + value_issues(raw) + missing_issues(raw)

In [15]:
def render(issues: list[Issue], raw: dict[str, pd.DataFrame]) -> str:
    """Markdown report: summary counts, then one table per category."""
    found = [i for i in issues if i.rows > 0]
    lines = [
        "# Data quality report",
        "",
        "Generated from the raw CSVs; nothing is typed by hand.",
        "",
        f"{len(issues)} checks run, {len(found)} found at least one row. "
        "Checks with 0 rows are listed too: they are the rules the validation in Part 2 enforces on every run.",
        "",
        "Raw row counts: " + ", ".join(f"{t} {len(df):,}" for t, df in raw.items()) + ".",
    ]
    for cat in dict.fromkeys(i.category for i in issues):
        lines += ["", f"## {cat}", "", "| Table | Check | Rows | Handling |", "| --- | --- | ---: | --- |"]
        lines += [f"| {i.table} | {i.check} | {i.rows:,} | {i.handling} |" for i in issues if i.category == cat]
    return "\n".join(lines) + "\n"

#### Run every check and write the report

In [16]:
raw = load_raw()
issues = find_issues(raw)
show(render(issues, raw))
# Checks that found at least one row:
pd.DataFrame([vars(i) for i in issues if i.rows]).drop(columns="handling")

# Data quality report

Generated from the raw CSVs; nothing is typed by hand.

42 checks run, 32 found at least one row. Checks with 0 rows are listed too: they are the rules the validation in Part 2 enforces on every run.

Raw row counts: customers 99,441, geolocation 1,000,163, order_items 112,650, order_payments 103,886, order_reviews 99,224, orders 99,441, products 32,951, sellers 3,095, category_translation 71.

## Impossible dates

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| orders | Delivered to customer before purchase | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| orders | Estimated delivery date before purchase | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| orders | Approved before purchase | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |

## Inverted dates

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| orders | Handed to carrier before purchase | 166 | Kept. Carrier and approval dates are not used in any metric. |
| orders | Handed to carrier before payment approval | 1,359 | Kept. Carrier and approval dates are not used in any metric. |
| orders | Delivered to customer before handed to carrier | 23 | Kept. Delivery time is measured from purchase, which is never after delivery. |
| order_reviews | Review created before the order's purchase date | 64 | Kept. The score is still used; review timing is not analysed. |

## Inconsistent status

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| orders | Status 'delivered' but no delivery date | 8 | Delay is NULL, so these orders drop out of late-delivery metrics. |
| orders | Not delivered (e.g. canceled) but has a delivery date | 6 | Delivery metrics set to NULL during cleaning (Part 2). |

## Out of window

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| orders | Purchased before 2017-01 or after 2018-08 | 349 | Kept in the database, excluded from every metric (in_window = 0). |

## Orphan rows

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| order_items | order_id not in orders | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| order_items | product_id not in products | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| order_items | seller_id not in sellers | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| order_payments | order_id not in orders | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| order_reviews | order_id not in orders | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| orders | customer_id not in customers | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |

## Missing children

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| orders | Order has no items | 775 | Kept. Revenue joins to items, so these orders add no revenue (767 of them are canceled/unavailable). |
| orders | Order has no payment | 1 | Kept. Revenue is item price, not payments. |
| orders | Order has no review | 768 | Kept. Review averages use COUNT(review_score), so unreviewed orders are not counted as 0. |

## Duplicate reviews

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| order_reviews | Extra reviews for an order that already has one | 551 | Keep the most recently answered review per order. |
| order_reviews | Rows whose review_id is shared with another order | 1,603 | Kept. 789 review_ids each cover several orders (one review for a multi-order purchase), so reviews are keyed by order_id. |

## Duplicate rows

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| geolocation | Fully duplicated rows | 261,831 | Collapsed to one median point per zip prefix. |

## Suspicious values

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| order_items | Price <= 0 | 0 | Checked in validation (Part 2): the pipeline stops if this is ever > 0. |
| order_items | Freight = 0 | 383 | Kept (free shipping is plausible). Revenue excludes freight anyway. |
| order_payments | payment_value <= 0 | 9 | Kept and logged, so payment totals still reconcile. |
| order_payments | payment_type 'not_defined' | 3 | Kept and logged. |
| order_payments | 0 instalments | 2 | Kept. Instalments only appear in the payment-mix average. |
| products | Weight = 0 g | 4 | Kept. Product weight is not used. |

## Unlabelled values

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| products | Category has no English translation | 2 | Two missing translations added by hand during cleaning (Part 2). |

## Missing values

| Table | Check | Rows | Handling |
| --- | --- | ---: | --- |
| order_reviews | `review_comment_title` is empty (88.3%) | 87,656 | Kept. Text is optional; only the score is used. |
| order_reviews | `review_comment_message` is empty (58.7%) | 58,247 | Kept. Text is optional; only the score is used. |
| orders | `order_approved_at` is empty (0.2%) | 160 | Kept. Not used in metrics. |
| orders | `order_delivered_carrier_date` is empty (1.8%) | 1,783 | Kept. Not used in metrics. |
| orders | `order_delivered_customer_date` is empty (3.0%) | 2,965 | Order drops out of delivery metrics (delay is NULL). |
| products | `product_category_name` is empty (1.9%) | 610 | Labelled 'unknown'. |
| products | `product_name_lenght` is empty (1.9%) | 610 | Kept. Column not used in metrics. |
| products | `product_description_lenght` is empty (1.9%) | 610 | Kept. Column not used in metrics. |
| products | `product_photos_qty` is empty (1.9%) | 610 | Kept. Column not used in metrics. |
| products | `product_weight_g` is empty (0.0%) | 2 | Kept. Column not used in metrics. |
| products | `product_length_cm` is empty (0.0%) | 2 | Kept. Column not used in metrics. |
| products | `product_height_cm` is empty (0.0%) | 2 | Kept. Column not used in metrics. |
| products | `product_width_cm` is empty (0.0%) | 2 | Kept. Column not used in metrics. |


,category,table,check,rows
0,Inverted dates,orders,Handed to carrier before purchase,166
1,Inverted dates,orders,Handed to carrier before payment approval,1359
2,Inverted dates,orders,Delivered to customer before handed to carrier,23
3,Inverted dates,order_reviews,Review created before the order's purchase date,64
4,Inconsistent status,orders,Status 'delivered' but no delivery date,8
5,Inconsistent status,orders,Not delivered (e.g. canceled) but has a delive...,6
6,Out of window,orders,Purchased before 2017-01 or after 2018-08,349
7,Missing children,orders,Order has no items,775
8,Missing children,orders,Order has no payment,1
9,Missing children,orders,Order has no review,768


## Part 2 · Clean, load into SQLite, validate

**Input:** `data/raw/*.csv`. **Output:** `data/olist.db` (every later part and the dashboard read it).

Steps:
1. Clean each table with vectorised pandas (rules listed below).
2. Build `order_summary`: one row per order, aggregating items and payments *before* joining.
3. Load everything into SQLite with indexes.
4. Run the validation checks on the result. **If any check fails, this notebook raises an error and
   `run_all.py` stops**, so no later step runs on bad data.

#### Assumptions and cleaning rules

1. **Who is a customer.** `customer_id` is created fresh for every order (99,441 ids for 99,441 orders). A person is `customer_unique_id` (96,096 distinct). All customer counts and repeat rates use `customer_unique_id`.
2. **Analysis window.** Business metrics use orders purchased 2017-01-01 to 2018-08-31 (`orders.in_window = 1`). 2016 has only ~330 orders and no orders in November; Sep and Oct 2018 have 20 orders between them. Including those months would show fake drops and spikes.
3. **What counts as a sale.** Orders with status `canceled` or `unavailable` are excluded from revenue and customer counts (`orders.is_sale = 0`). Other statuses (shipped, invoiced, processing, etc.) count as sales.
4. **Revenue.** Revenue = sum of `order_items.price`. Freight is excluded. Payment value is not used for revenue because it includes freight and card-instalment interest. 775 orders have no items at all (mostly `unavailable`); they have no revenue. 5 in-window sale orders have no items and therefore drop out of revenue queries.
5. **Late delivery.** `delay_days` = delivered date minus estimated date, in whole days (positive = late). The estimate has no time of day, so the delivered timestamp is truncated to its date first. `is_late = 1` if `delay_days > 0`. Orders without a delivery date have NULL metrics. Extreme values (-147 to +188 days) are kept; they are real records and are grouped into buckets in the analysis.
6. **Contradictory rows.** 6 orders are `canceled` but carry a delivery date. Their delivery metrics are set to NULL. 8 `delivered` orders have no delivery date; they have NULL delivery metrics and are left out of delivery analysis.
7. **Reviews.** 547 orders have more than one review (551 extra rows), and in 202 of those the scores differ. We keep the **latest answered** review per order, on the view that it reflects the customer's final opinion. 814 extra rows reuse a `review_id` already used for another order; the score is identical in every such case, so they are kept as-is.
8. **Categories.** Portuguese names are translated with the provided table. Two categories missing from it were translated manually: `pc_gamer` -> `pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos` -> `portable_kitchen_and_food_preparers`. 610 products have no category and are labelled `unknown`.
9. **Column-name typos.** `product_name_lenght` and `product_description_lenght` are renamed to `..._length`.
10. **Payments.** 9 payment rows have `payment_value <= 0` (6 voucher, 3 `not_defined`). They are kept so totals reconcile with the source. One order has no payment row.
11. **Geolocation.** 1,000,163 rows with many repeats per zip prefix were collapsed to 19,015 rows (median lat/lng per prefix). Not used in the queries; kept for maps.
12. **Zip prefixes** are read as text so leading zeros survive.
13. **Joins.** Items and payments are aggregated to one row per order before joining (see `order_summary` and the CTEs in `queries.sql`), so orders with several items aren't counted several times.

#### Tables

##### orders (99,441 rows, one per order)
| Column | Meaning |
| --- | --- |
| order_id | Order key |
| customer_id | Per-order customer key; join to `customers` to get the person |
| order_status | delivered, shipped, canceled, unavailable, invoiced, processing, created, approved |
| order_purchase_timestamp | When the order was placed |
| order_approved_at | Payment approval time (160 NULL) |
| order_delivered_carrier_date | Handed to carrier (1,783 NULL) |
| order_delivered_customer_date | Delivered to customer (2,965 NULL) |
| order_estimated_delivery_date | Date promised to the customer at purchase |
| delivery_days | *Derived.* Days from purchase to delivery (fractional) |
| delay_days | *Derived.* Delivered date minus estimated date, whole days; > 0 means late |
| is_late | *Derived.* 1 late, 0 on time or early, NULL if not delivered |
| purchase_month | *Derived.* `YYYY-MM` of purchase |
| in_window | *Derived.* 1 if purchased 2017-01-01 .. 2018-08-31 |
| is_sale | *Derived.* 0 if canceled or unavailable, else 1 |

##### customers (99,441 rows, one per customer_id)
| Column | Meaning |
| --- | --- |
| customer_id | Per-order key (1:1 with orders) |
| customer_unique_id | The person; use for repeat-customer analysis |
| customer_zip_code_prefix | First 5 digits of zip, text |
| customer_city | Lower-cased, trimmed |
| customer_state | Two-letter state code |

##### order_items (112,650 rows, one per item line)
| Column | Meaning |
| --- | --- |
| order_id, order_item_id | Key; `order_item_id` is the line number within the order |
| product_id, seller_id | What was sold and by whom; one order can have several sellers |
| shipping_limit_date | Deadline for the seller to hand to the carrier |
| price | Item price (R$); revenue is the sum of this |
| freight_value | Freight charged for this item (R$) |

##### order_payments (103,886 rows)
| Column | Meaning |
| --- | --- |
| order_id, payment_sequential | Key; an order can be split over several payments |
| payment_type | credit_card, boleto, voucher, debit_card, not_defined |
| payment_installments | Number of instalments (credit card) |
| payment_value | Amount (R$), includes freight |

##### order_reviews (98,673 rows after cleaning, one per order)
| Column | Meaning |
| --- | --- |
| review_id | Review key (not unique across orders, see rule 7) |
| order_id | Order reviewed |
| review_score | 1-5 stars |
| review_comment_title, review_comment_message | Free text, mostly NULL (88% / 59%) |
| review_creation_date, review_answer_timestamp | When the survey was sent / answered |

##### products (32,951 rows)
| Column | Meaning |
| --- | --- |
| product_id | Key |
| product_category_name | Original Portuguese category (610 NULL) |
| product_category | *Derived.* English category, `unknown` if missing |
| product_name_length, product_description_length, product_photos_qty | Listing attributes (610 NULL) |
| product_weight_g, product_length_cm, product_height_cm, product_width_cm | Package size (2 NULL) |

##### sellers (3,095 rows)
`seller_id`, `seller_zip_code_prefix` (text), `seller_city`, `seller_state`.

##### geolocation (19,015 rows)
`geolocation_zip_code_prefix`, median `lat` and `lng`, `state`.

##### category_translation (71 rows)
`product_category_name` (Portuguese) -> `product_category_name_english`, as provided by Olist.

##### order_summary (99,441 rows, one per order)
All `orders` columns plus `customer_unique_id`, `customer_state`, `n_items`, `item_value` (sum of price), `freight_value`, `payment_value` (sum over payment rows) and `review_score` (latest review). Built by aggregating each child table to one row per order before joining; the script asserts the row count equals `orders`.

### Cleaning rules

Each function below handles one table. Row counts before and after are printed so every change can be checked.

In [17]:
# Analysis window: the 20 complete, well-populated months. 2016 has a
# two-month gap and only ~330 orders; Sep/Oct 2018 have 20 orders in total.
WINDOW_START = pd.Timestamp("2017-01-01")
WINDOW_END = pd.Timestamp("2018-09-01")  # exclusive

# Two Portuguese categories are missing from the official translation table.
EXTRA_TRANSLATIONS = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_and_food_preparers",
}

# Order statuses that never became a sale; excluded from revenue metrics.
NON_SALE_STATUSES = ("canceled", "unavailable")

log_lines: list[str] = []

In [18]:
def log(msg: str) -> None:
    """Print a message and keep it for the cleaning log."""
    print(msg)
    log_lines.append(f"- {msg}")

In [19]:
def read_raw(table: str) -> pd.DataFrame:
    """Read a raw CSV, keeping zip prefixes as text and parsing dates."""
    df = pd.read_csv(
        RAW_DIR / RAW_FILES[table],
        dtype=STRING_COLUMNS,
        encoding="utf-8-sig",
    )
    for col in DATE_COLUMNS.get(table, []):
        df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

In [20]:
def clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise city names; customer_id is unique so nothing is dropped."""
    df = df.copy()
    df["customer_city"] = df["customer_city"].str.strip().str.lower()
    df["customer_state"] = df["customer_state"].str.strip().str.upper()
    return df

In [21]:
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    """Add delivery metrics, purchase month and window/sale flags."""
    df = df.copy()
    delivered = df["order_delivered_customer_date"]
    df["delivery_days"] = (delivered - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
    # Positive = late. Estimated date is a date at midnight, so compare dates.
    df["delay_days"] = (delivered.dt.normalize() - df["order_estimated_delivery_date"]).dt.days
    df["is_late"] = np.where(delivered.isna(), np.nan, (df["delay_days"] > 0).astype(float))
    df["purchase_month"] = df["order_purchase_timestamp"].dt.strftime("%Y-%m")
    df["in_window"] = (
        (df["order_purchase_timestamp"] >= WINDOW_START) & (df["order_purchase_timestamp"] < WINDOW_END)
    ).astype(int)
    df["is_sale"] = (~df["order_status"].isin(NON_SALE_STATUSES)).astype(int)
    # Delivered timestamps on non-delivered orders (6 canceled rows) are
    # contradictory; blank the derived metrics so they don't enter delivery stats.
    bad = (df["order_status"] != "delivered") & delivered.notna()
    log(f"orders: {int(bad.sum())} non-delivered orders carry a delivery date; delivery metrics set to NULL")
    df.loc[bad, ["delivery_days", "delay_days", "is_late"]] = np.nan
    return df

In [22]:
def clean_products(df: pd.DataFrame, translation: pd.DataFrame) -> pd.DataFrame:
    """Fix column typos, translate categories, label missing ones 'unknown'."""
    df = df.rename(
        columns={
            "product_name_lenght": "product_name_length",
            "product_description_lenght": "product_description_length",
        }
    )
    mapping = dict(
        zip(translation["product_category_name"], translation["product_category_name_english"], strict=False)
    )
    mapping.update(EXTRA_TRANSLATIONS)
    df["product_category"] = df["product_category_name"].map(mapping)
    n_missing = int(df["product_category_name"].isna().sum())
    df["product_category"] = df["product_category"].fillna("unknown")
    log(f"products: {n_missing} products with no category labelled 'unknown'")
    return df

In [23]:
def clean_reviews(df: pd.DataFrame) -> pd.DataFrame:
    """Keep one review per order: the most recently answered one."""
    before = len(df)
    df = df.sort_values(["order_id", "review_answer_timestamp"])
    df = df.drop_duplicates(subset="order_id", keep="last")
    log(f"order_reviews: {before:,} -> {len(df):,} rows (kept latest review per order)")
    return df

In [24]:
def clean_payments(df: pd.DataFrame) -> pd.DataFrame:
    """Flag odd rows instead of dropping them so totals stay reconcilable."""
    df = df.copy()
    zero = df.loc[df["payment_value"] <= 0, "payment_type"].value_counts().to_dict()
    log(f"order_payments: {sum(zero.values())} rows with payment_value <= 0 kept, by type: {zero}")
    log(f"order_payments: {int((df['payment_type'] == 'not_defined').sum())} rows with payment_type 'not_defined' kept")
    return df

In [25]:
def clean_geolocation(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse to one row per zip prefix using the median coordinate."""
    before = len(df)
    out = df.groupby("geolocation_zip_code_prefix", as_index=False).agg(
        lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"), state=("geolocation_state", "first")
    )
    log(f"geolocation: {before:,} -> {len(out):,} rows (one median point per zip prefix)")
    return out

In [26]:
def build_order_summary(
    orders: pd.DataFrame, items: pd.DataFrame, payments: pd.DataFrame, reviews: pd.DataFrame, customers: pd.DataFrame
) -> pd.DataFrame:
    """One row per order, aggregating items and payments BEFORE joining.

    Aggregating first avoids the row multiplication you get from joining
    items (many per order) and payments (many per order) at the same time.
    """
    item_agg = items.groupby("order_id", as_index=False).agg(
        n_items=("order_item_id", "count"),
        item_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
    )
    pay_agg = payments.groupby("order_id", as_index=False).agg(payment_value=("payment_value", "sum"))
    out = (
        orders.merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left")
        .merge(item_agg, on="order_id", how="left")
        .merge(pay_agg, on="order_id", how="left")
        .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
    )
    assert len(out) == len(orders), "order_summary must keep one row per order"
    return out

#### Clean every table

In [27]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
raw = {t: read_raw(t) for t in RAW_FILES}
for t, df in raw.items():
    log(f"{t}: {len(df):,} raw rows")

translation = raw["category_translation"]
tables = {
    "customers": clean_customers(raw["customers"]),
    "orders": clean_orders(raw["orders"]),
    "order_items": raw["order_items"],
    "order_payments": clean_payments(raw["order_payments"]),
    "order_reviews": clean_reviews(raw["order_reviews"]),
    "products": clean_products(raw["products"], translation),
    "sellers": raw["sellers"],
    "geolocation": clean_geolocation(raw["geolocation"]),
    "category_translation": translation,
}
tables["order_summary"] = build_order_summary(
    tables["orders"],
    tables["order_items"],
    tables["order_payments"],
    tables["order_reviews"],
    tables["customers"],
)
log(
    f"orders: {int(tables['orders']['in_window'].sum()):,} orders inside analysis window "
    f"{WINDOW_START.date()} to {(WINDOW_END - pd.Timedelta(days=1)).date()}"
)

customers: 99,441 raw rows
geolocation: 1,000,163 raw rows
order_items: 112,650 raw rows
order_payments: 103,886 raw rows
order_reviews: 99,224 raw rows
orders: 99,441 raw rows
products: 32,951 raw rows
sellers: 3,095 raw rows
category_translation: 71 raw rows
orders: 6 non-delivered orders carry a delivery date; delivery metrics set to NULL


order_payments: 9 rows with payment_value <= 0 kept, by type: {'voucher': 6, 'not_defined': 3}
order_payments: 3 rows with payment_type 'not_defined' kept


order_reviews: 99,224 -> 98,673 rows (kept latest review per order)
products: 610 products with no category labelled 'unknown'
geolocation: 1,000,163 -> 19,015 rows (one median point per zip prefix)


orders: 99,092 orders inside analysis window 2017-01-01 to 2018-08-31


#### Load into SQLite

Timestamps are stored as ISO text, which SQLite's date functions understand.

In [28]:
if DB_PATH.exists():
    DB_PATH.unlink()
with sqlite3.connect(DB_PATH) as con:
    for name, df in tables.items():
        df = df.copy()
        # Store timestamps as ISO text, which SQLite date functions understand.
        for col in df.select_dtypes(include=["datetime"]).columns:
            df[col] = df[col].dt.strftime("%Y-%m-%d %H:%M:%S")
        df.to_sql(name, con, index=False)
        log(f"loaded {name}: {len(df):,} rows")
    con.executescript(
        """
        CREATE INDEX idx_orders_id ON orders(order_id);
        CREATE INDEX idx_orders_cust ON orders(customer_id);
        CREATE INDEX idx_items_order ON order_items(order_id);
        CREATE INDEX idx_items_product ON order_items(product_id);
        CREATE INDEX idx_items_seller ON order_items(seller_id);
        CREATE INDEX idx_pay_order ON order_payments(order_id);
        CREATE INDEX idx_rev_order ON order_reviews(order_id);
        CREATE INDEX idx_cust_id ON customers(customer_id);
        CREATE INDEX idx_prod_id ON products(product_id);
        """
    )
print(f"Database written to {DB_PATH}")

loaded customers: 99,441 rows


loaded orders: 99,441 rows


loaded order_items: 112,650 rows


loaded order_payments: 103,886 rows


loaded order_reviews: 98,673 rows
loaded products: 32,951 rows


loaded sellers: 3,095 rows
loaded geolocation: 19,015 rows
loaded category_translation: 71 rows


loaded order_summary: 99,441 rows


Database written to data\olist.db


---
### Validation

Each check is a plain pandas expression that counts offending rows; every count must be 0.

In [29]:
Tables = dict[str, pd.DataFrame]
Check = tuple[str, Callable[[Tables], int]]

In [30]:
def _ts(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

In [31]:
def duplicated_keys(df: pd.DataFrame, cols: list[str]) -> int:
    """Number of rows whose key appears more than once."""
    return int(df.duplicated(subset=cols, keep=False).sum())

In [32]:
def missing_parent(child: pd.DataFrame, col: str, parent: pd.DataFrame, parent_col: str | None = None) -> int:
    """Number of child rows whose foreign key has no parent row."""
    return int((~child[col].isin(parent[parent_col or col])).sum())

In [33]:
CHECKS: list[Check] = [
    # Values
    ("order_items.price > 0", lambda t: int((t["order_items"]["price"] <= 0).sum())),
    ("order_items.freight_value >= 0", lambda t: int((t["order_items"]["freight_value"] < 0).sum())),
    ("order_payments.payment_value >= 0", lambda t: int((t["order_payments"]["payment_value"] < 0).sum())),
    (
        "order_reviews.review_score between 1 and 5",
        lambda t: int((~t["order_reviews"]["review_score"].between(1, 5)).sum()),
    ),
    (
        "orders.is_late is 0, 1 or NULL",
        lambda t: int((~t["orders"]["is_late"].isin([0, 1]) & t["orders"]["is_late"].notna()).sum()),
    ),
    # Dates
    (
        "delivered date >= purchase date",
        lambda t: int(
            (_ts(t["orders"]["order_delivered_customer_date"]) < _ts(t["orders"]["order_purchase_timestamp"])).sum()
        ),
    ),
    (
        "approved date >= purchase date",
        lambda t: int((_ts(t["orders"]["order_approved_at"]) < _ts(t["orders"]["order_purchase_timestamp"])).sum()),
    ),
    (
        "estimated delivery date >= purchase date",
        lambda t: int(
            (
                _ts(t["orders"]["order_estimated_delivery_date"])
                < _ts(t["orders"]["order_purchase_timestamp"]).dt.normalize()
            ).sum()
        ),
    ),
    ("delivery_days >= 0 when present", lambda t: int((t["orders"]["delivery_days"] < 0).sum())),
    (
        "only delivered orders have delivery metrics",
        lambda t: int(((t["orders"]["order_status"] != "delivered") & t["orders"]["delay_days"].notna()).sum()),
    ),
    # Unique keys
    ("orders.order_id unique", lambda t: duplicated_keys(t["orders"], ["order_id"])),
    ("customers.customer_id unique", lambda t: duplicated_keys(t["customers"], ["customer_id"])),
    ("products.product_id unique", lambda t: duplicated_keys(t["products"], ["product_id"])),
    ("sellers.seller_id unique", lambda t: duplicated_keys(t["sellers"], ["seller_id"])),
    (
        "order_items (order_id, order_item_id) unique",
        lambda t: duplicated_keys(t["order_items"], ["order_id", "order_item_id"]),
    ),
    (
        "order_payments (order_id, payment_sequential) unique",
        lambda t: duplicated_keys(t["order_payments"], ["order_id", "payment_sequential"]),
    ),
    ("order_reviews: one review per order", lambda t: duplicated_keys(t["order_reviews"], ["order_id"])),
    ("order_summary: one row per order", lambda t: duplicated_keys(t["order_summary"], ["order_id"])),
    (
        "geolocation: one row per zip prefix",
        lambda t: duplicated_keys(t["geolocation"], ["geolocation_zip_code_prefix"]),
    ),
    # Foreign keys
    ("orders.customer_id -> customers", lambda t: missing_parent(t["orders"], "customer_id", t["customers"])),
    ("order_items.order_id -> orders", lambda t: missing_parent(t["order_items"], "order_id", t["orders"])),
    ("order_items.product_id -> products", lambda t: missing_parent(t["order_items"], "product_id", t["products"])),
    ("order_items.seller_id -> sellers", lambda t: missing_parent(t["order_items"], "seller_id", t["sellers"])),
    ("order_payments.order_id -> orders", lambda t: missing_parent(t["order_payments"], "order_id", t["orders"])),
    ("order_reviews.order_id -> orders", lambda t: missing_parent(t["order_reviews"], "order_id", t["orders"])),
    # Completeness
    ("products.product_category never NULL", lambda t: int(t["products"]["product_category"].isna().sum())),
    ("order_summary row count == orders row count", lambda t: abs(len(t["order_summary"]) - len(t["orders"]))),
]

In [34]:
def run_checks(tables: Tables, checks: list[Check] = CHECKS) -> list[tuple[str, int]]:
    """Return (check name, offending rows) for every check."""
    return [(name, fn(tables)) for name, fn in checks]

In [35]:
def failures(results: list[tuple[str, int]]) -> list[tuple[str, int]]:
    """Only the checks that found at least one bad row."""
    return [(n, c) for n, c in results if c > 0]

In [36]:
def load_tables() -> Tables:
    """Read every table the checks need from the SQLite database."""
    names = [
        "orders",
        "customers",
        "products",
        "sellers",
        "order_items",
        "order_payments",
        "order_reviews",
        "order_summary",
        "geolocation",
    ]
    with sqlite3.connect(DB_PATH) as con:
        return {n: pd.read_sql_query(f"SELECT * FROM {n}", con) for n in names}

#### Run the checks

In [37]:
results = run_checks(load_tables())
bad = failures(results)
for name, n in results:
    print(f"{'FAIL' if n else 'ok  '}  {name}" + (f"  ({n:,} rows)" if n else ""))
if bad:
    raise RuntimeError(f"DATA VALIDATION FAILED: {len(bad)} of {len(results)} checks broken.")
print(f"\nAll {len(results)} validation checks passed.")

ok    order_items.price > 0
ok    order_items.freight_value >= 0
ok    order_payments.payment_value >= 0
ok    order_reviews.review_score between 1 and 5
ok    orders.is_late is 0, 1 or NULL
ok    delivered date >= purchase date
ok    approved date >= purchase date
ok    estimated delivery date >= purchase date
ok    delivery_days >= 0 when present
ok    only delivered orders have delivery metrics
ok    orders.order_id unique
ok    customers.customer_id unique
ok    products.product_id unique
ok    sellers.seller_id unique
ok    order_items (order_id, order_item_id) unique
ok    order_payments (order_id, payment_sequential) unique
ok    order_reviews: one review per order
ok    order_summary: one row per order
ok    geolocation: one row per zip prefix
ok    orders.customer_id -> customers
ok    order_items.order_id -> orders
ok    order_items.product_id -> products
ok    order_items.seller_id -> sellers
ok    order_payments.order_id -> orders
ok    order_reviews.order_id -> orders
ok  

## Part 3 · SQL analysis, headline metrics and metric definitions

**Input:** `data/olist.db`. **Output:** the query results (also saved to `data/query_results/`), the headline metrics and the metric definitions page.

The SQL lives in `queries.sql`, not in this notebook, because the dashboard runs the same queries:

- the analysis queries (monthly revenue, late vs on-time, states, sellers, cohorts...).
- one canonical query per business metric, each with a `-- DEFINITION:` comment.

`run_query(name)` (defined at the top) runs a query by name.

### 1. Run every analysis query

Each result is printed and saved as a CSV.

In [38]:
RESULTS_DIR = DATA / "query_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
with sqlite3.connect(DB_PATH) as con:
    for name in load_queries():
        df = run_query(name, con)
        df.to_csv(RESULTS_DIR / f"{name}.csv", index=False)
        print(f"\n=== {name} ({len(df)} rows)")
        print(df.to_string(index=False))


=== monthly_revenue (20 rows)
purchase_month  orders    revenue  running_revenue  mom_growth_pct
       2017-01     787  120098.27        120098.27             NaN
       2017-02    1718  244959.35        365057.62           104.0
       2017-03    2617  368341.32        733398.94            50.4
       2017-04    2377  353842.98       1087241.92            -3.9
       2017-05    3640  503159.19       1590401.11            42.2
       2017-06    3205  429916.61       2020317.72           -14.6
       2017-07    3946  492287.30       2512605.02            14.5
       2017-08    4272  568245.79       3080850.81            15.4
       2017-09    4227  621415.91       3702266.72             9.4
       2017-10    4547  660179.62       4362446.34             6.2
       2017-11    7421 1003862.14       5366308.48            52.1
       2017-12    5618  742183.79       6108492.27           -26.1
       2018-01    7187  945456.29       7053948.56            27.4
       2018-02    6624  837895.


=== top_categories (15 rows)
 revenue_rank              category  orders    revenue  revenue_share_pct  cumulative_share_pct
            1         health_beauty    8757 1251145.54               9.30                  9.30
            2         watches_gifts    5599 1194824.97               8.88                 18.19
            3        bed_bath_table    9394 1035485.07               7.70                 25.89
            4        sports_leisure    7656  977728.77               7.27                 33.15
            5 computers_accessories    6636  902922.70               6.71                 39.87
            6       furniture_decor    6373  721584.27               5.37                 45.23
            7            housewares    5838  625538.73               4.65                 49.88
            8            cool_stuff    3609  619724.39               4.61                 54.49
            9                  auto    3863  585177.48               4.35                 58.84
          


=== delay_vs_review (6 rows)
     delay_bucket  delivered_orders  reviewed_orders  avg_review  pct_1_or_2_stars
 1. 7+ days early             75882            75487       4.314               9.0
2. 0-6 days early             13790            13695       4.166              10.7
 3. 1-3 days late              1869             1851       3.291              32.1
 4. 4-7 days late              1802             1748       2.103              67.7
5. 8-14 days late              1478             1446       1.671              80.2
 6. 15+ days late              1382             1333       1.725              78.3



=== late_vs_ontime (2 rows)
delivery  delivered_orders  pct_of_orders  reviewed_orders  avg_review     revenue
    late              6531           6.79             6378       2.271   985618.47
 on_time             89672          93.21            89182       4.291 12194159.48



=== repeat_customer_rate (2 rows)
           customer_key  customers  repeat_customers  repeat_rate_pct  avg_orders_per_customer
     customer_unique_id      94707              2875             3.04                   1.0338
customer_id (wrong key)      97910                 0             0.00                   1.0000



=== worst_sellers (10 rows)
 worst_rank                        seller_id seller_state  delivered_orders  reviewed_orders  avg_review  late_rate_pct  sellers_with_50_plus
          1 1ca7077d890b907f89be8c954a02686a           SP               108              107       2.393           16.7                   424
          2 2eb70248d66e0e3ef83659f71b244378           SP               187              184       2.793           11.2                   424
          3 972d0f9cf61b499a4812cf0bfa3ad3c4           SC                81               79       3.025           11.1                   424
          4 a49928bcdf77c55c6d6e05e09a9b4ca5           SP                96               96       3.031           21.9                   424
          5 54965bbe3e4f07ae045b90b0b8541f52           PR                73               69       3.145           30.1                   424
          6 bbad7e518d7af88a0897397ffdca1979           SP                68               67       3.179           16.2

... (3 outputs left out)


=== revenue (1 rows)
      value  orders
13449529.68   97905



=== active_customers (1 rows)
 value
 94707



=== repeat_customers (1 rows)
 value  active_customers  rate_pct
  2875             94707      3.04

=== late_delivery_rate (1 rows)
 value  late_orders  delivered_orders
  6.79         6531             96203



=== avg_review_score (1 rows)
 value  reviewed_orders
 4.156            95568


### 2. Headline numbers

Every figure quoted in the memo and the README is produced here (or appears directly in a
query result above), so the documents can be checked against a run.

In [39]:
def compute_metrics(con: sqlite3.Connection) -> dict[str, float]:
    """Return a flat dict of headline metrics."""
    m: dict[str, float] = {}

    monthly = run_query("monthly_revenue", con)
    rev_2017 = monthly.loc[monthly["purchase_month"].between("2017-01", "2017-08"), "revenue"].sum()
    rev_2018 = monthly.loc[monthly["purchase_month"].between("2018-01", "2018-08"), "revenue"].sum()
    m["total_revenue_window"] = monthly["revenue"].sum()
    m["total_orders_window"] = monthly["orders"].sum()
    m["revenue_jan_aug_2017"] = rev_2017
    m["revenue_jan_aug_2018"] = rev_2018
    m["yoy_growth_jan_aug_pct"] = 100 * (rev_2018 / rev_2017 - 1)
    m["avg_monthly_revenue_2018"] = monthly.loc[monthly["purchase_month"] >= "2018-01", "revenue"].mean()

    late = run_query("late_vs_ontime", con).set_index("delivery")
    m["late_share_pct"] = late.loc["late", "pct_of_orders"]
    m["late_orders"] = late.loc["late", "delivered_orders"]
    m["late_avg_review"] = late.loc["late", "avg_review"]
    m["ontime_avg_review"] = late.loc["on_time", "avg_review"]
    m["review_gap_late_vs_ontime"] = late.loc["on_time", "avg_review"] - late.loc["late", "avg_review"]
    m["late_revenue"] = late.loc["late", "revenue"]
    m["late_revenue_share_pct"] = 100 * late.loc["late", "revenue"] / late["revenue"].sum()

    repeat = run_query("repeat_customer_rate", con).set_index("customer_key")
    m["repeat_rate_pct"] = repeat.loc["customer_unique_id", "repeat_rate_pct"]
    m["customers"] = repeat.loc["customer_unique_id", "customers"]

    states = run_query("state_delivery", con)
    m["state_late_vs_review_corr"] = states["late_rate_pct"].corr(states["avg_review"])
    rj = states.set_index("customer_state").loc["RJ"]
    sp = states.set_index("customer_state").loc["SP"]
    m["rj_late_rate_pct"], m["rj_orders"] = rj["late_rate_pct"], rj["delivered_orders"]
    m["sp_late_rate_pct"] = sp["late_rate_pct"]

    # Seller concentration of bad reviews: share of all 1-2 star reviews (window,
    # delivered) that come from the 10% of 50+ order sellers with the lowest scores.
    seller = pd.read_sql_query(
        """
        WITH so AS (SELECT DISTINCT seller_id, order_id FROM order_items)
        SELECT so.seller_id, COUNT(*) AS n, AVG(r.review_score) AS avg_review,
               COALESCE(SUM(r.review_score <= 2), 0) AS low_reviews
        FROM so JOIN orders o ON o.order_id = so.order_id
        LEFT JOIN order_reviews r ON r.order_id = so.order_id
        WHERE o.in_window = 1 AND o.order_status = 'delivered'
        GROUP BY so.seller_id
        """,
        con,
    )
    big = seller[seller["n"] >= 50].sort_values("avg_review")
    bottom = big.head(int(round(len(big) * 0.10)))
    m["sellers_50_plus"] = len(big)
    m["bottom10pct_sellers"] = len(bottom)
    m["bottom10pct_share_of_orders_pct"] = 100 * bottom["n"].sum() / seller["n"].sum()
    m["bottom10pct_share_of_low_reviews_pct"] = 100 * bottom["low_reviews"].sum() / seller["low_reviews"].sum()
    return m

In [40]:
with sqlite3.connect(DB_PATH) as con:
    metrics = compute_metrics(con)
lines = [
    "# Headline metrics",
    "",
    "Window: Jan 2017 - Aug 2018.",
    "",
    "| Metric | Value |",
    "| --- | --- |",
]
for k, v in metrics.items():
    lines.append(f"| {k} | {v:,.4g} |" if abs(v) < 1000 else f"| {k} | {v:,.2f} |")
show("\n".join(lines) + "\n")

# Headline metrics

Window: Jan 2017 - Aug 2018.

| Metric | Value |
| --- | --- |
| total_revenue_window | 13,449,529.68 |
| total_orders_window | 97,905.00 |
| revenue_jan_aug_2017 | 3,080,850.81 |
| revenue_jan_aug_2018 | 7,341,037.41 |
| yoy_growth_jan_aug_pct | 138.3 |
| avg_monthly_revenue_2018 | 917,629.68 |
| late_share_pct | 6.79 |
| late_orders | 6,531.00 |
| late_avg_review | 2.271 |
| ontime_avg_review | 4.291 |
| review_gap_late_vs_ontime | 2.02 |
| late_revenue | 985,618.47 |
| late_revenue_share_pct | 7.478 |
| repeat_rate_pct | 3.04 |
| customers | 94,707.00 |
| state_late_vs_review_corr | -0.9066 |
| rj_late_rate_pct | 12.1 |
| rj_orders | 12,310.00 |
| sp_late_rate_pct | 4.5 |
| sellers_50_plus | 424 |
| bottom10pct_sellers | 42 |
| bottom10pct_share_of_orders_pct | 6.61 |
| bottom10pct_share_of_low_reviews_pct | 13.1 |


### 3. Metric definitions page

The page below shows each metric's definition, its exact SQL and its current value. The text
and SQL are read from `queries.sql`, so the page cannot drift from the queries the dashboard runs.

In [41]:
TITLES = {
    "revenue": ("Revenue", "R$ {value:,.2f} from {orders:,} orders"),
    "active_customers": ("Active customer", "{value:,} active customers"),
    "repeat_customers": ("Repeat customer", "{value:,} repeat customers = {rate_pct}% of {active_customers:,} active"),
    "late_delivery_rate": ("Late delivery", "{value}% late ({late_orders:,} of {delivered_orders:,} delivered orders)"),
    "avg_review_score": ("Average review score", "{value} stars over {reviewed_orders:,} reviewed delivered orders"),
}

In [42]:
def split_definition(sql: str) -> tuple[str, str]:
    """Separate the '-- DEFINITION:' comment block from the SQL body."""
    comment, body = [], []
    for line in sql.splitlines():
        (comment if line.startswith("--") else body).append(line)
    text = " ".join(re.sub(r"^--\s*(DEFINITION:)?\s*", "", c) for c in comment).strip()
    return text, "\n".join(body).strip()

In [43]:
def render(con: sqlite3.Connection) -> str:
    """Build the markdown page."""
    queries = load_queries()
    lines = [
        "# Metric definitions",
        "",
        "Generated from the metric queries in `queries.sql`. "
        "The dashboard's KPI tiles run these same queries.",
        "",
        "All metrics use the analysis window: orders purchased 2017-01-01 to 2018-08-31. "
        "A customer is a `customer_unique_id` (a person), never `customer_id` (new on every order).",
        "",
        "| Metric | Current value |",
        "| --- | --- |",
    ]
    sections = []
    for name, (title, fmt) in TITLES.items():
        row = run_metric(name, con).iloc[0].to_dict()
        row = {k: int(v) if float(v).is_integer() else v for k, v in row.items()}
        value = fmt.format(**row)
        lines.append(f"| {title} | {value} |")
        definition, body = split_definition(queries[name])
        sections += ["", f"## {title}", "", definition, "", f"Current value: **{value}**", "", "```sql", body, "```"]
    return "\n".join(lines + sections) + "\n"

In [44]:
with sqlite3.connect(DB_PATH) as con:
    page = render(con)
show(page)

# Metric definitions

Generated from the metric queries in `queries.sql`. The dashboard's KPI tiles run these same queries.

All metrics use the analysis window: orders purchased 2017-01-01 to 2018-08-31. A customer is a `customer_unique_id` (a person), never `customer_id` (new on every order).

| Metric | Current value |
| --- | --- |
| Revenue | R$ 13,449,529.68 from 97,905 orders |
| Active customer | 94,707 active customers |
| Repeat customer | 2,875 repeat customers = 3.04% of 94,707 active |
| Late delivery | 6.79% late (6,531 of 96,203 delivered orders) |
| Average review score | 4.156 stars over 95,568 reviewed delivered orders |

## Revenue

Sum of item prices (excluding freight) on orders purchased in the window whose status is not 'canceled' or 'unavailable'. In Brazilian reais (R$).

Current value: **R$ 13,449,529.68 from 97,905 orders**

```sql
SELECT ROUND(SUM(oi.price), 2) AS value,
       COUNT(DISTINCT o.order_id) AS orders
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
WHERE o.in_window = 1 AND o.is_sale = 1;
```

## Active customer

Number of distinct people (customer_unique_id, not customer_id) with at least one sale order (not canceled/unavailable) purchased in the period. For a single month, add `AND o.purchase_month = 'YYYY-MM'`.

Current value: **94,707 active customers**

```sql
SELECT COUNT(DISTINCT c.customer_unique_id) AS value
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.in_window = 1 AND o.is_sale = 1;
```

## Repeat customer

Active customers with 2 or more sale orders in the window. The rate is repeat customers divided by active customers. Two orders on the same day count as two.

Current value: **2,875 repeat customers = 3.04% of 94,707 active**

```sql
WITH per_person AS (
    SELECT c.customer_unique_id, COUNT(*) AS n_orders
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.in_window = 1 AND o.is_sale = 1
    GROUP BY c.customer_unique_id
)
SELECT SUM(n_orders >= 2)                              AS value,
       COUNT(*)                                        AS active_customers,
       ROUND(100.0 * SUM(n_orders >= 2) / COUNT(*), 2) AS rate_pct
FROM per_person;
```

## Late delivery

Share of delivered orders (status 'delivered' with a delivery date) whose delivery date falls on a later calendar day than the estimated delivery date shown to the customer. Delivered on the estimated day counts as on time. In percent.

Current value: **6.79% late (6,531 of 96,203 delivered orders)**

```sql
SELECT ROUND(100.0 * AVG(is_late), 2) AS value,
       SUM(is_late = 1)               AS late_orders,
       COUNT(*)                       AS delivered_orders
FROM orders
WHERE in_window = 1 AND order_status = 'delivered' AND is_late IS NOT NULL;
```

## Average review score

Mean review score (1-5) over delivered orders in the window that have a review. One review per order (the latest answered). Orders without a review are left out, not counted as zero.

Current value: **4.156 stars over 95,568 reviewed delivered orders**

```sql
SELECT ROUND(AVG(r.review_score), 3) AS value,
       COUNT(r.review_score)         AS reviewed_orders
FROM orders o
JOIN order_reviews r ON r.order_id = o.order_id
WHERE o.in_window = 1 AND o.order_status = 'delivered';
```


## Part 4 · Cohort retention


A cohort is the month of a person's first sale order. For each cohort we measure the share of customers
who ordered again *k* calendar months later. The query is `cohort_retention` in `queries.sql`; the
dashboard's cohort page draws its heatmap from the same query.

### Helper functions

In [45]:
def retention_matrix(long: pd.DataFrame) -> pd.DataFrame:
    """Cohort x month-offset table of retention %, NaN where not yet observable."""
    return long.pivot(index="cohort_month", columns="month_offset", values="retention_pct")

In [46]:
def summarise(long: pd.DataFrame) -> dict[str, float]:
    """Customer-weighted retention figures used in the report."""
    sizes = long[long["month_offset"] == 0].set_index("cohort_month")["cohort_customers"]
    last_month = long["cohort_month"].max()
    months_observed = {c: (pd.Period(last_month) - pd.Period(c)).n for c in sizes.index}
    out: dict[str, float] = {"cohorts": len(sizes), "customers": int(sizes.sum())}
    for k in (1, 3, 6):
        eligible = [c for c, n in months_observed.items() if n >= k]
        active = long[(long["month_offset"] == k) & long["cohort_month"].isin(eligible)]["active_customers"].sum()
        out[f"month_{k}_retention_pct"] = 100 * active / sizes.loc[eligible].sum()
        out[f"month_{k}_cohorts"] = len(eligible)
    peak = long[long["month_offset"] > 0]
    out["max_cell_pct"] = float(peak["retention_pct"].max())
    best = peak.loc[peak["retention_pct"].idxmax()]
    out["max_cell"] = [best["cohort_month"], int(best["month_offset"])]
    return out

In [47]:
def ever_returned(con: sqlite3.Connection, within_months: int = 6) -> pd.DataFrame:
    """Share of each cohort that placed any second order within N months (fully observed cohorts only)."""
    long = run_query("cohort_retention", con)
    first = run_query_first_orders(con)
    last = pd.Period(long["cohort_month"].max())
    first["cohort"] = first["first_month"].map(pd.Period)
    first = first[first["cohort"].map(lambda p: (last - p).n >= within_months)]
    first["returned"] = first["months_to_second"].between(0, within_months)
    first["same_month"] = first["months_to_second"] == 0
    return first.groupby(first["cohort"].astype(str)).agg(
        customers=("returned", "size"), returned=("returned", "sum"), same_month=("same_month", "sum")
    )

In [48]:
def run_query_first_orders(con: sqlite3.Connection) -> pd.DataFrame:
    """Each person's first order month and calendar months until their second order (NaN if none)."""
    sql = """
    WITH o AS (
        SELECT c.customer_unique_id AS person, o.order_purchase_timestamp AS ts, o.purchase_month AS m
        FROM orders o JOIN customers c ON c.customer_id = o.customer_id
        WHERE o.in_window = 1 AND o.is_sale = 1
    ),
    ranked AS (
        SELECT person, m, ROW_NUMBER() OVER (PARTITION BY person ORDER BY ts) AS rn FROM o
    )
    SELECT a.person, a.m AS first_month, b.m AS second_month
    FROM ranked a LEFT JOIN ranked b ON b.person = a.person AND b.rn = 2
    WHERE a.rn = 1
    """
    df = pd.read_sql_query(sql, con)
    df["months_to_second"] = _month_index(df["second_month"]) - _month_index(df["first_month"])
    return df

In [49]:
def _month_index(months: pd.Series) -> pd.Series:
    """'YYYY-MM' -> year * 12 + month (NaN stays NaN)."""
    return months.str[:4].astype(float) * 12 + months.str[5:7].astype(float)

In [50]:
def render(long: pd.DataFrame, s: dict, returned: pd.DataFrame) -> str:
    """Markdown report with the heatmap numbers and an honest reading."""
    mat = retention_matrix(long)
    cols = [c for c in mat.columns if c <= 12]
    header = "| Cohort | Customers | " + " | ".join(f"M{c}" for c in cols if c > 0) + " |"
    sizes = long[long["month_offset"] == 0].set_index("cohort_month")["cohort_customers"]
    rows = [header, "| --- | ---: | " + " | ".join("---:" for c in cols if c > 0) + " |"]
    for cohort, r in mat.iterrows():
        cells = ["" if pd.isna(r.get(c)) else f"{r[c]:.2f}" for c in cols if c > 0]
        rows.append(f"| {cohort} | {sizes[cohort]:,} | " + " | ".join(cells) + " |")
    ret_total = 100 * returned["returned"].sum() / returned["customers"].sum()
    same_month = 100 * returned["same_month"].sum() / returned["customers"].sum()
    return "\n".join(
        [
            "# Cohort retention",
            "",
            "Generated from the `cohort_retention` query. The dashboard's "
            "*Cohort retention* page shows the same numbers as a heatmap.",
            "",
            "A cohort is the month of a person's first sale order (Jan 2017 - Aug 2018). "
            "Cell M*k* = % of the cohort with at least one sale order *k* calendar months later. "
            "Blank cells are months the data does not reach yet.",
            "",
            "## Summary",
            "",
            f"- {s['cohorts']} cohorts, {s['customers']:,} customers.",
            f"- Month-1 retention: **{s['month_1_retention_pct']:.2f}%** "
            f"(customer-weighted over the {s['month_1_cohorts']} cohorts with a month 1).",
            f"- Month-3: {s['month_3_retention_pct']:.2f}%, month-6: {s['month_6_retention_pct']:.2f}% "
            f"({s['month_3_cohorts']} and {s['month_6_cohorts']} cohorts observable).",
            f"- The single highest cell after month 0 is {s['max_cell_pct']:.2f}% "
            f"(cohort {s['max_cell'][0]}, month {int(s['max_cell'][1])}).",
            f"- Within 6 calendar months of their first order (same month included), **{ret_total:.2f}%** of "
            f"customers placed a second order; {same_month:.2f}% did so in the same month as the first "
            f"(cohorts {returned.index.min()} to {returned.index.max()}, which can all be followed for 6 months).",
            "",
            "## What this means",
            "",
            "- Olist is almost entirely a one-purchase marketplace in this data: no cohort keeps even 1% "
            "of its customers active in any later month. The heatmap is mostly empty for that reason, not "
            "because of a bug.",
            "- Differences between cohorts are tiny in absolute terms (a few customers per cell), so "
            "month-to-month variation in the heatmap is mostly noise. Don't read trends into it.",
            "- Possible reasons, none testable here: many purchases are one-off categories (furniture, "
            "gifts), customers may come back through another marketplace, and the public dataset is a "
            "sample, which can hide repeat orders that fell outside it.",
            "- So retention is a growth *opportunity* to test (memo recommendation 3), not something the "
            "current business relies on.",
            "",
            "## Retention by cohort (%, months 1-12)",
            "",
            *rows,
            "",
        ]
    )

#### Compute and write the report

In [51]:
with sqlite3.connect(DB_PATH) as con:
    long = run_query("cohort_retention", con)
    returned = ever_returned(con)
s = summarise(long)
show(render(long, s, returned))
for k, v in s.items():
    print(k, v)

# Cohort retention

Generated from the `cohort_retention` query. The dashboard's *Cohort retention* page shows the same numbers as a heatmap.

A cohort is the month of a person's first sale order (Jan 2017 - Aug 2018). Cell M*k* = % of the cohort with at least one sale order *k* calendar months later. Blank cells are months the data does not reach yet.

## Summary

- 20 cohorts, 94,707 customers.
- Month-1 retention: **0.48%** (customer-weighted over the 19 cohorts with a month 1).
- Month-3: 0.26%, month-6: 0.23% (17 and 14 cohorts observable).
- The single highest cell after month 0 is 0.71% (cohort 2017-10, month 1).
- Within 6 calendar months of their first order (same month included), **3.28%** of customers placed a second order; 1.60% did so in the same month as the first (cohorts 2017-01 to 2018-02, which can all be followed for 6 months).

## What this means

- Olist is almost entirely a one-purchase marketplace in this data: no cohort keeps even 1% of its customers active in any later month. The heatmap is mostly empty for that reason, not because of a bug.
- Differences between cohorts are tiny in absolute terms (a few customers per cell), so month-to-month variation in the heatmap is mostly noise. Don't read trends into it.
- Possible reasons, none testable here: many purchases are one-off categories (furniture, gifts), customers may come back through another marketplace, and the public dataset is a sample, which can hide repeat orders that fell outside it.
- So retention is a growth *opportunity* to test (memo recommendation 3), not something the current business relies on.

## Retention by cohort (%, months 1-12)

| Cohort | Customers | M1 | M2 | M3 | M4 | M5 | M6 | M7 | M8 | M9 | M10 | M11 | M12 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 2017-01 | 753 | 0.40 | 0.27 | 0.13 | 0.40 | 0.13 | 0.40 | 0.13 | 0.13 |  | 0.40 | 0.13 | 0.66 |
| 2017-02 | 1,690 | 0.24 | 0.30 | 0.12 | 0.41 | 0.12 | 0.24 | 0.18 | 0.12 | 0.18 | 0.12 | 0.30 | 0.12 |
| 2017-03 | 2,571 | 0.51 | 0.35 | 0.39 | 0.35 | 0.16 | 0.16 | 0.31 | 0.35 | 0.08 | 0.39 | 0.12 | 0.23 |
| 2017-04 | 2,326 | 0.60 | 0.21 | 0.17 | 0.30 | 0.26 | 0.34 | 0.30 | 0.30 | 0.17 | 0.26 | 0.09 | 0.09 |
| 2017-05 | 3,541 | 0.48 | 0.48 | 0.40 | 0.31 | 0.34 | 0.42 | 0.14 | 0.25 | 0.25 | 0.25 | 0.34 | 0.25 |
| 2017-06 | 3,102 | 0.45 | 0.35 | 0.39 | 0.26 | 0.39 | 0.35 | 0.23 | 0.13 | 0.23 | 0.32 | 0.32 | 0.16 |
| 2017-07 | 3,823 | 0.52 | 0.34 | 0.23 | 0.29 | 0.21 | 0.31 | 0.10 | 0.18 | 0.26 | 0.21 | 0.29 | 0.13 |
| 2017-08 | 4,130 | 0.68 | 0.34 | 0.27 | 0.36 | 0.53 | 0.29 | 0.27 | 0.14 | 0.14 | 0.24 | 0.19 | 0.12 |
| 2017-09 | 4,076 | 0.69 | 0.54 | 0.29 | 0.44 | 0.22 | 0.22 | 0.24 | 0.29 | 0.20 | 0.27 | 0.07 |  |
| 2017-10 | 4,392 | 0.71 | 0.25 | 0.09 | 0.23 | 0.20 | 0.20 | 0.36 | 0.27 | 0.20 | 0.20 |  |  |
| 2017-11 | 7,191 | 0.56 | 0.39 | 0.17 | 0.20 | 0.18 | 0.11 | 0.20 | 0.12 | 0.06 |  |  |  |
| 2017-12 | 5,439 | 0.22 | 0.28 | 0.35 | 0.26 | 0.20 | 0.17 | 0.04 | 0.22 |  |  |  |  |
| 2018-01 | 6,952 | 0.33 | 0.36 | 0.29 | 0.29 | 0.16 | 0.17 | 0.23 |  |  |  |  |  |
| 2018-02 | 6,357 | 0.36 | 0.39 | 0.30 | 0.27 | 0.22 | 0.20 |  |  |  |  |  |  |
| 2018-03 | 6,932 | 0.40 | 0.30 | 0.29 | 0.13 | 0.12 |  |  |  |  |  |  |  |
| 2018-04 | 6,698 | 0.58 | 0.31 | 0.24 | 0.13 |  |  |  |  |  |  |  |  |
| 2018-05 | 6,588 | 0.52 | 0.26 | 0.20 |  |  |  |  |  |  |  |  |  |
| 2018-06 | 5,921 | 0.42 | 0.27 |  |  |  |  |  |  |  |  |  |  |
| 2018-07 | 6,016 | 0.52 |  |  |  |  |  |  |  |  |  |  |  |
| 2018-08 | 6,209 |  |  |  |  |  |  |  |  |  |  |  |  |


cohorts 20
customers 94707
month_1_retention_pct 0.4824967795882393
month_1_cohorts 19
month_3_retention_pct 0.2586173116861065
month_3_cohorts 17
month_6_retention_pct 0.22895479473936425
month_6_cohorts 14
max_cell_pct 0.706
max_cell ['2017-10', 1]


#### Retention matrix (% of cohort active k months later, months 1-12)

In [52]:
retention_matrix(long).loc[:, 1:12].round(2)

month_offset,1,2,3,4,5,6,7,8,9,10,11,12
cohort_month,,,,,,,,,,,,
2017-01,0.40,0.27,0.13,0.40,0.13,0.40,0.13,0.13,NaN,0.40,0.13,0.66
2017-02,0.24,0.30,0.12,0.41,0.12,0.24,0.18,0.12,0.18,0.12,0.30,0.12
2017-03,0.51,0.35,0.39,0.35,0.16,0.16,0.31,0.35,0.08,0.39,0.12,0.23
2017-04,0.60,0.22,0.17,0.30,0.26,0.34,0.30,0.30,0.17,0.26,0.09,0.09
2017-05,0.48,0.48,0.40,0.31,0.34,0.42,0.14,0.25,0.25,0.25,0.34,0.25
2017-06,0.45,0.36,0.39,0.26,0.39,0.36,0.23,0.13,0.23,0.32,0.32,0.16
2017-07,0.52,0.34,0.24,0.29,0.21,0.31,0.10,0.18,0.26,0.21,0.29,0.13
2017-08,0.68,0.34,0.27,0.36,0.53,0.29,0.27,0.14,0.14,0.24,0.19,0.12
2017-09,0.69,0.54,0.29,0.44,0.22,0.22,0.24,0.29,0.20,0.27,0.07,NaN


## Part 5 · Sizing the memo recommendations


Each estimate combines something measured in the data (counts, rates, order values) with an explicit
assumption from `ASSUMPTIONS` below. They are **estimates, not forecasts**: they treat associations
(late delivery → bad review) as if fixing the cause would remove the effect, which historical data
cannot prove.

### Assumptions and sizing functions

In [53]:
ASSUMPTIONS = {
    # Rec 1: states worse than SP are brought down to SP's late rate.
    "benchmark_state": "SP",
    # Rec 2: bottom-10% sellers improve to the median low-review rate of 50+ order sellers.
    "seller_min_orders": 50,
    "seller_bottom_share": 0.10,
    # Rec 3: the second-order offer lifts the 90-day repeat rate of the target group by these
    # absolute percentage points (scenarios, not measurements), at this discount.
    "repeat_lift_pp": [0.5, 1.0, 2.0],
    "offer_discount": 0.10,
    "repeat_window_days": 90,
}

In [54]:
def load(con: sqlite3.Connection) -> pd.DataFrame:
    """Order-level table for in-window sale orders."""
    df = pd.read_sql_query(
        """SELECT order_id, customer_unique_id, customer_state, order_status, order_purchase_timestamp,
                  item_value, review_score, is_late
           FROM order_summary WHERE in_window = 1 AND is_sale = 1""",
        con,
    )
    df["ts"] = pd.to_datetime(df["order_purchase_timestamp"])
    return df

In [55]:
def size_late_deliveries(df: pd.DataFrame) -> dict:
    """Rec 1: fewer late orders -> fewer 1-2 star reviews and (maybe) more repeat buyers."""
    d = df[(df["order_status"] == "delivered") & df["is_late"].notna()]
    bench = d.loc[d["customer_state"] == ASSUMPTIONS["benchmark_state"], "is_late"].mean()
    by_state = d.groupby("customer_state").agg(orders=("is_late", "size"), late_rate=("is_late", "mean"))
    excess = (by_state["late_rate"] - bench).clip(lower=0) * by_state["orders"]
    avoided = float(excess.sum())

    rev = d[d["review_score"].notna()]
    low_late = (rev.loc[rev["is_late"] == 1, "review_score"] <= 2).mean()
    low_ontime = (rev.loc[rev["is_late"] == 0, "review_score"] <= 2).mean()

    # Repeat behaviour by whether the customer's FIRST order was late.
    first = d.sort_values("ts").groupby("customer_unique_id").head(1)
    later = df.groupby("customer_unique_id")["order_id"].count()
    first = first.assign(repeat=first["customer_unique_id"].map(later) >= 2)
    rep_late = first.loc[first["is_late"] == 1, "repeat"].mean()
    rep_ontime = first.loc[first["is_late"] == 0, "repeat"].mean()
    aov = df["item_value"].mean()
    return {
        "benchmark_late_rate_pct": 100 * bench,
        "late_orders_now": int(d["is_late"].sum()),
        "late_orders_avoided": avoided,
        "top_states_by_avoided": excess.sort_values(ascending=False).head(5).round(0).to_dict(),
        "low_review_rate_late_pct": 100 * low_late,
        "low_review_rate_ontime_pct": 100 * low_ontime,
        "low_reviews_avoided": avoided * (low_late - low_ontime),
        "low_reviews_now": int((rev["review_score"] <= 2).sum()),
        "repeat_rate_first_late_pct": 100 * rep_late,
        "repeat_rate_first_ontime_pct": 100 * rep_ontime,
        "extra_repeat_customers": avoided * max(rep_ontime - rep_late, 0),
        "avg_order_value": aov,
        "revenue_from_extra_repeats": avoided * max(rep_ontime - rep_late, 0) * aov,
    }

In [56]:
def size_sellers(con: sqlite3.Connection) -> dict:
    """Rec 2: bottom sellers improve to the median low-review rate."""
    s = pd.read_sql_query(
        """WITH so AS (SELECT DISTINCT seller_id, order_id FROM order_items)
           SELECT so.seller_id, COUNT(*) AS n, COUNT(r.review_score) AS reviewed,
                  AVG(r.review_score) AS avg_review, COALESCE(SUM(r.review_score <= 2), 0) AS low
           FROM so JOIN orders o ON o.order_id = so.order_id
           LEFT JOIN order_reviews r ON r.order_id = so.order_id
           WHERE o.in_window = 1 AND o.order_status = 'delivered'
           GROUP BY so.seller_id""",
        con,
    )
    big = s[s["n"] >= ASSUMPTIONS["seller_min_orders"]].copy()
    big["low_rate"] = big["low"] / big["reviewed"]
    bottom = big.sort_values("avg_review").head(int(round(len(big) * ASSUMPTIONS["seller_bottom_share"])))
    median_rate = big["low_rate"].median()
    avoided = float(((bottom["low_rate"] - median_rate).clip(lower=0) * bottom["reviewed"]).sum())
    return {
        "bottom_sellers": len(bottom),
        "bottom_low_rate_pct": 100 * bottom["low"].sum() / bottom["reviewed"].sum(),
        "median_low_rate_pct": 100 * median_rate,
        "low_reviews_bottom_now": int(bottom["low"].sum()),
        "low_reviews_avoided": avoided,
        "all_low_reviews": int(s["low"].sum()),
        "share_of_all_low_reviews_avoided_pct": 100 * avoided / s["low"].sum(),
    }

In [57]:
def size_second_purchase(df: pd.DataFrame) -> dict:
    """Rec 3: offer after a good first experience; revenue under assumed lifts."""
    days = ASSUMPTIONS["repeat_window_days"]
    end = df["ts"].max()
    df = df.sort_values("ts")
    df["order_no"] = df.groupby("customer_unique_id").cumcount() + 1
    first = df[df["order_no"] == 1].set_index("customer_unique_id")
    second = df[df["order_no"] == 2].set_index("customer_unique_id")["ts"]
    first = first.assign(second_ts=second)
    # Only customers whose 90-day window is fully inside the data.
    first = first[first["ts"] <= end - pd.Timedelta(days=days)]
    good = first[(first["order_status"] == "delivered") & (first["is_late"] == 0) & (first["review_score"] >= 4)]
    came_back = (good["second_ts"] - good["ts"]).dt.days.between(1, days)
    second_orders = df[df["order_no"] == 2]
    aov2 = second_orders["item_value"].mean()
    per_year = len(good) * 365 / (good["ts"].max() - good["ts"].min()).days
    scenarios = []
    for lift in ASSUMPTIONS["repeat_lift_pp"]:
        extra = per_year * lift / 100
        scenarios.append(
            {
                "lift_pp": lift,
                "extra_orders_per_year": extra,
                "net_revenue_per_year": extra * aov2 * (1 - ASSUMPTIONS["offer_discount"]),
            }
        )
    return {
        "target_customers": len(good),
        "target_customers_per_year": per_year,
        "baseline_90d_repeat_pct": 100 * came_back.mean(),
        "avg_second_order_value": aov2,
        "scenarios": scenarios,
    }

In [58]:
def render(r1: dict, r2: dict, r3: dict) -> str:
    """Markdown report."""
    rows3 = "\n".join(
        f"| +{s['lift_pp']} pp | {s['extra_orders_per_year']:,.0f} | R$ {s['net_revenue_per_year']:,.0f} |"
        for s in r3["scenarios"]
    )
    states = ", ".join(f"{k} {v:,.0f}" for k, v in r1["top_states_by_avoided"].items())
    return f"""# Sizing the memo recommendations (ESTIMATES)

Generated from the database. Every number below is computed from the database; the
assumptions are listed in `ASSUMPTIONS` and repeated here.
These are **estimates, not forecasts**: they assume the observed associations are causal,
which historical data cannot show. Use them to rank the ideas, then test each one.

## 1. Bring late-delivery rates down to {ASSUMPTIONS["benchmark_state"]}'s level

Assumption: every state with a higher late rate than {ASSUMPTIONS["benchmark_state"]} \
({r1["benchmark_late_rate_pct"]:.2f}%) is brought down to it; nothing else changes.

| | Estimate |
| --- | ---: |
| Late orders today (20 months) | {r1["late_orders_now"]:,} |
| Late orders avoided | {r1["late_orders_avoided"]:,.0f} |
| Largest contributions (orders) | {states} |
| 1-2 star rate: late vs on time | {r1["low_review_rate_late_pct"]:.1f}% vs {r1["low_review_rate_ontime_pct"]:.1f}% |
| 1-2 star reviews avoided | {r1["low_reviews_avoided"]:,.0f} of {r1["low_reviews_now"]:,} |
| Repeat rate when first order late vs on time | {r1["repeat_rate_first_late_pct"]:.2f}% vs {r1["repeat_rate_first_ontime_pct"]:.2f}% |
| Extra repeat customers | {r1["extra_repeat_customers"]:,.0f} |
| Revenue from them (x avg order value R$ {r1["avg_order_value"]:,.2f}) | R$ {r1["revenue_from_extra_repeats"]:,.0f} |

The review effect is large; the revenue effect through repeat purchases is small, because
repeat buying is rare whether or not the first order was late.

## 2. Performance plan for the bottom 10% of sellers

Assumption: the {r2["bottom_sellers"]} lowest-rated sellers (50+ delivered orders) reach the median
1-2 star rate of all 50+ order sellers ({r2["median_low_rate_pct"]:.1f}%, vs {r2["bottom_low_rate_pct"]:.1f}% today).

| | Estimate |
| --- | ---: |
| 1-2 star reviews from these sellers today | {r2["low_reviews_bottom_now"]:,} |
| 1-2 star reviews avoided | {r2["low_reviews_avoided"]:,.0f} |
| Share of all 1-2 star reviews (window, delivered) | {r2["share_of_all_low_reviews_avoided_pct"]:.1f}% of {r2["all_low_reviews"]:,} |

## 3. Second-purchase offer after a good first order

Target group: customers whose first order was delivered on time and reviewed 4-5 stars, and
whose next {ASSUMPTIONS["repeat_window_days"]} days are fully inside the data: {r3["target_customers"]:,} customers
(about {r3["target_customers_per_year"]:,.0f} a year). Their measured {ASSUMPTIONS["repeat_window_days"]}-day repeat rate
today is **{r3["baseline_90d_repeat_pct"]:.2f}%**. Average second-order value: R$ {r3["avg_second_order_value"]:,.2f}.

The lift is an assumption (no experiment exists yet), so three scenarios are shown, net of a
{100 * ASSUMPTIONS["offer_discount"]:.0f}% discount:

| Assumed lift in 90-day repeat rate | Extra orders / year | Net revenue / year |
| --- | ---: | ---: |
{rows3}

For scale: 2018 revenue averaged about R$ 918k a month (headline metrics, Part 3).
"""

#### Run the three estimates

In [59]:
with sqlite3.connect(DB_PATH) as con:
    df = load(con)
    r1, r2, r3 = size_late_deliveries(df), size_sellers(con), size_second_purchase(df)
show(render(r1, r2, r3))
print(json.dumps({"late": r1, "sellers": r2, "second_purchase": r3}, indent=2, default=float))

# Sizing the memo recommendations (ESTIMATES)

Generated from the database. Every number below is computed from the database; the
assumptions are listed in `ASSUMPTIONS` and repeated here.
These are **estimates, not forecasts**: they assume the observed associations are causal,
which historical data cannot show. Use them to rank the ideas, then test each one.

## 1. Bring late-delivery rates down to SP's level

Assumption: every state with a higher late rate than SP (4.50%) is brought down to it; nothing else changes.

| | Estimate |
| --- | ---: |
| Late orders today (20 months) | 6,531 |
| Late orders avoided | 2,234 |
| Largest contributions (orders) | RJ 941, BA 250, SC 132, ES 124, CE 119 |
| 1-2 star rate: late vs on time | 62.4% vs 9.2% |
| 1-2 star reviews avoided | 1,188 of 12,228 |
| Repeat rate when first order late vs on time | 2.58% vs 3.12% |
| Extra repeat customers | 12 |
| Revenue from them (x avg order value R$ 137.37) | R$ 1,644 |

The review effect is large; the revenue effect through repeat purchases is small, because
repeat buying is rare whether or not the first order was late.

## 2. Performance plan for the bottom 10% of sellers

Assumption: the 42 lowest-rated sellers (50+ delivered orders) reach the median
1-2 star rate of all 50+ order sellers (12.0%, vs 26.4% today).

| | Estimate |
| --- | ---: |
| 1-2 star reviews from these sellers today | 1,686 |
| 1-2 star reviews avoided | 919 |
| Share of all 1-2 star reviews (window, delivered) | 7.1% of 12,867 |

## 3. Second-purchase offer after a good first order

Target group: customers whose first order was delivered on time and reviewed 4-5 stars, and
whose next 90 days are fully inside the data: 56,599 customers
(about 40,428 a year). Their measured 90-day repeat rate
today is **1.33%**. Average second-order value: R$ 122.47.

The lift is an assumption (no experiment exists yet), so three scenarios are shown, net of a
10% discount:

| Assumed lift in 90-day repeat rate | Extra orders / year | Net revenue / year |
| --- | ---: | ---: |
| +0.5 pp | 202 | R$ 22,280 |
| +1.0 pp | 404 | R$ 44,560 |
| +2.0 pp | 809 | R$ 89,121 |

For scale: 2018 revenue averaged about R$ 918k a month (headline metrics, Part 3).


{
  "late": {
    "benchmark_late_rate_pct": 4.4976360801009925,
    "late_orders_now": 6531,
    "late_orders_avoided": 2233.720611896334,
    "top_states_by_avoided": {
      "RJ": 941.0,
      "BA": 250.0,
      "SC": 132.0,
      "ES": 124.0,
      "CE": 119.0
    },
    "low_review_rate_late_pct": 62.41768579492004,
    "low_review_rate_ontime_pct": 9.24738175865085,
    "low_reviews_avoided": 1187.6760406660933,
    "low_reviews_now": 12228,
    "repeat_rate_first_late_pct": 2.58349086326402,
    "repeat_rate_first_ontime_pct": 3.1193802738968044,
    "extra_repeat_customers": 11.970272222274293,
    "avg_order_value": 137.37326673816455,
    "revenue_from_extra_repeats": 1644.395398918928
  },
  "sellers": {
    "bottom_sellers": 42,
    "bottom_low_rate_pct": 26.368470441038472,
    "median_low_rate_pct": 12.0,
    "low_reviews_bottom_now": 1686,
    "low_reviews_avoided": 918.72,
    "all_low_reviews": 12867,
    "share_of_all_low_reviews_avoided_pct": 7.140125903474003
  },
 

## Part 6 · Memo: three actions for Olist marketplace performance

**To:** Head of Marketplace Operations
**Data:** 97,905 orders purchased Jan 2017 - Aug 2018 (canceled/unavailable excluded). All numbers come from the outputs above. Impact figures are **estimates** with stated assumptions; details in Part 5.

### Context

Revenue grew 138.3% (Jan-Aug 2018 vs Jan-Aug 2017) but has been flat through 2018 at R$838k-994k a month. Growth now has to come from keeping customers happy and getting them to come back, not only from new buyers.

### 1. Cut late deliveries, starting with Rio de Janeiro and the Northeast

- 6.79% of delivered orders (6,531) arrived after the promised date. Late orders average **2.27 stars vs 4.29** for on-time orders.
- Orders 4-7 days late get a 1-2 star review **67.7%** of the time, versus 9.0% for orders arriving 7+ days early.
- Late orders hold **R$986k (7.48%)** of delivered-order revenue, so this is the revenue most exposed to poor reviews.
- RJ is the largest single problem: 12,310 orders with a **12.1% late rate vs 4.5% in SP**. AL (21.5%) and MA (17.5%) are worst by rate. Across states, late rate and average review correlate at r = -0.91.

**Action:** review carrier performance on routes into RJ and the Northeast, and for those routes set the promised date from actual delivery times instead of the current estimate. Track the late rate per state every week.

**Estimated impact (estimate, assumes the association is causal):** if every state did as well as SP (4.50% late), about **2,234 of 6,531 late orders** would arrive on time (RJ alone accounts for ~941). With 1-2 star rates of 62.4% for late vs 9.2% for on-time orders, that is about **1,188 fewer 1-2 star reviews** (of 12,228). The revenue effect is small: customers whose first order was late come back almost as often (2.58% vs 3.12%), so the gain through repeat purchases is only about 12 customers and R$1.6k. Treat this as a customer-experience fix, not a revenue lever.

### 2. Put the lowest-rated sellers on a performance plan

- 424 sellers have 50+ delivered orders. The bottom 10% of them by review score (42 sellers) handle **6.61% of seller orders but produce 13.1% of all 1-2 star reviews**, about twice their share.
- Lateness is only part of it: the 10 worst sellers have late rates from 6.5% to 30.1%, so product or listing quality likely plays a role (this data can't separate the two).

**Action:** a monthly seller scorecard (average review, late rate, volume), with coaching for sellers in the bottom 10% and limits on visibility for those who don't improve within a set period.

**Estimated impact (estimate):** if the 42 bottom sellers reached the median 1-2 star rate of 50+ order sellers (12.0%, vs 26.4% today), about **919 fewer 1-2 star reviews**, or 7.1% of all of them, from 42 sellers.

### 3. Build a second-purchase programme

- Only **3.04%** of customers (2,875 of 94,707) bought more than once in 20 months. Counting by `customer_id` instead would report 0%, which is why the analysis uses `customer_unique_id`.
- With so few repeat buyers, flat 2018 revenue means acquisition is carrying the business.

**Action:** after an on-time delivery with a 4-5 star review (the customers most likely to return), send a time-limited offer for a second order. Run it as an A/B test against a holdout group and measure repeat rate after 90 days. As a scale reference only: one percentage point of repeat rate is about 947 more returning customers on this base.

**Estimated impact (estimate, the lift is assumed, not measured):** about 40,400 customers a year have an on-time, 4-5 star first order, and only **1.33%** of them order again within 90 days today. At an average second order of R$122 and a 10% discount, a lift of +0.5 / +1 / +2 percentage points is worth about **R$22k / R$45k / R$89k a year** net. Small against ~R$918k of monthly revenue, which is why it should be tested cheaply before scaling.

### Which first?

By size of estimated effect: seller performance plans and late-delivery fixes mainly protect reviews (roughly 900-1,200 fewer 1-2 star reviews each); the second-purchase offer is the only one with a direct revenue estimate, and it is modest. None of the three is a large revenue lever on its own.

### Caveats

- These are associations in historical data, not proof of cause. Late delivery and low reviews could share a cause (for example a region or product type).
- Revenue is item price only (no freight) and amounts are in Brazilian reais.
- The data is a public, anonymised sample of Olist orders, not the full company.